# P3-01 · D-FINE T4 feasibility (512 vs 640, AMP)

Runs `notebooks/p3/p3_01_dfine_feasibility.py` on a **synthetic** COCO set (single class `tb_lesion`).
No real images and no sealed data are touched.

**Kaggle settings:** Accelerator = *GPU T4* (if you get T4 ×2, only GPU 0 is used), Internet = **On**
(needed for the git clone and the ImageNet-pretrained HGNetV2 backbone download).

Measures, per model variant × input size: peak GPU memory per batch size at the largest multi-scale
training size (512 → 640 px, 640 → 800 px), the largest workable batch, the chosen batch
(peak reserved ≤ 14 GB), and seconds per epoch.

When it finishes, paste the printed summary (or the `.csv` / `.json`) back so `docs/p3_dfine_feasibility.md` can be filled in.

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # one T4

REPO_URL = "https://github.com/<org>/cv_proj.git"   # TODO: team repo URL (private repo: use a Kaggle secret token)
BRANCH = "p3/P3-01-dfine-feasibility"
DFINE_COMMIT = "956d1709314c2c6a4df6f34de232054578a7449f"
WORK = "/kaggle/working/cv_proj"

In [ ]:
!git clone --recurse-submodules -b $BRANCH $REPO_URL $WORK
%cd $WORK
# Fallback if the submodule is not registered yet: clone D-FINE at the pinned commit.
!test -f third_party/D-FINE/train.py || (git clone https://github.com/Peterande/D-FINE third_party/D-FINE && git -C third_party/D-FINE checkout $DFINE_COMMIT)
!git -C third_party/D-FINE rev-parse HEAD

In [ ]:
# Kaggle already ships CUDA torch/torchvision; install only P3 extras.
!pip install -q -r requirements/p3.txt
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
import torch; print(torch.__version__, torch.version.cuda, torch.cuda.get_device_name(0))

`--n-train`: number of images in the real detector training set, used only to extrapolate s/epoch.
Set it from the C01 train split when P1 has published it; otherwise leave `N_TRAIN = None`.
Add `--models s` to run a single variant if the session is short on time.

In [ ]:
N_TRAIN = None
extra = f"--n-train {N_TRAIN}" if N_TRAIN else ""
!python notebooks/p3/p3_01_dfine_feasibility.py --models s m --sizes 512 640 --workers 4 --seed 0 {extra} --out /kaggle/working/p3_feasibility

In [ ]:
import glob, json, pandas as pd
latest = sorted(glob.glob("/kaggle/working/p3_feasibility/dfine_t4_*.csv"))[-1]
print(latest)
display(pd.read_csv(latest))
print(json.dumps(json.load(open(latest.replace(".csv", ".json")))["env"], indent=1))